# Predicting the **goal difference** with a Gaussian — a teaching notebook

A deliberately simple, end-to-end Bayesian workflow. Instead of modelling each
team's goals with a Poisson (what the main project does), here we model **one
number per match — the goal difference** — with a **Gaussian**.

**Why the goal difference?**
$$\text{goal\_diff} = \text{home\_goals} - \text{away\_goals}$$
The *sign* is the result, and it is the thing we actually care about:

| goal_diff | meaning |
|---|---|
| $> 0$ | **home** team won |
| $= 0$ | draw |
| $< 0$ | **away** team won (a negative margin) |

So a single continuous target already encodes *who won* and *by how much*.

**Why a Gaussian?** Goal difference is a signed quantity that is roughly
symmetric and bell-shaped around a small positive number (home edge). A Normal
likelihood is the simplest honest choice for a signed, continuous-ish target —
and it makes the whole model a clean Bayesian linear model, ideal for teaching.

We will go: **read data → define the target → write the Bayesian model → fit →
check → interpret → predict a fixture.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

from wc2026.data.sources import build_real_matches

RNG = 2026
plt.rcParams["figure.figsize"] = (8, 4)

## 1. Read the data

We use real international results (2022 → today), already filtered to the 48
World Cup teams by the project's loader. Each row is one played match.

In [ ]:
matches = build_real_matches(start="2022-01-01", end="2026-07-11")
matches = matches[["date", "home_team", "away_team", "home_goals", "away_goals"]].copy()
print(f"{len(matches)} matches, {matches['home_team'].nunique()} home teams")
matches.head()

## 2. The target variable: goal difference

We compute the target and look at its distribution. Watch the **sign**: the mass
just right of zero is the home advantage.

In [ ]:
matches["goal_diff"] = matches["home_goals"] - matches["away_goals"]

print(matches["goal_diff"].describe().round(2))

ax = matches["goal_diff"].plot.hist(bins=range(-8, 9), edgecolor="white")
ax.axvline(0, color="k", lw=1, ls="--")
ax.set_xlabel("goal difference  (home − away)")
ax.set_title("Target: signed goal difference  (<0 = away won,  >0 = home won)")
plt.show()

**What this tells us.** The histogram is roughly **symmetric and
bell-shaped**, centred a little above zero (the home edge). That is exactly the
shape a **Gaussian** describes well — which justifies our modelling choice. It is
not perfect (goal difference is integer-valued and has slightly heavy tails), but
for a simple, interpretable model the Normal is a very reasonable approximation.

## 3. Encode teams as integers

The model refers to teams by an index, so we map each team name to a column
number and build integer arrays for the home side, away side, and the target.

In [ ]:
teams = sorted(set(matches["home_team"]) | set(matches["away_team"]))
team_idx = {t: i for i, t in enumerate(teams)}
n_teams = len(teams)

home_i = matches["home_team"].map(team_idx).to_numpy()
away_i = matches["away_team"].map(team_idx).to_numpy()
y = matches["goal_diff"].to_numpy().astype(float)

print(f"{n_teams} teams, {len(y)} observations")

## 4. The Bayesian model (the heart of the notebook)

We tell a small **generative story**: *how would nature produce a goal
difference?*

Give every team a single latent number, its **strength** $s_t$. When team $h$
hosts team $a$, the *expected* margin is the home team's strength minus the away
team's, plus a small **home advantage** $\alpha$. The actual margin scatters
around that expectation with noise $\sigma$.

**Likelihood (one row of data):**
$$\text{goal\_diff}_g \;\sim\; \mathcal{N}\!\big(\mu_g,\; \sigma\big),
\qquad \mu_g = \alpha + s_{h(g)} - s_{a(g)}$$

**Priors (what we believe before seeing data):**
$$
s_t \sim \mathcal{N}(0,\ \tau) \quad\text{(team strengths, shared scale }\tau)\\
\alpha \sim \mathcal{N}(0.3,\ 0.5) \quad\text{(home edge, mild positive)}\\
\sigma \sim \text{HalfNormal}(3) \quad\text{(match-to-match noise, positive)}\\
\tau \sim \text{HalfNormal}(1) \quad\text{(how spread out team strengths are)}
$$

**Why each piece:**

- **$\mu_g = \alpha + s_h - s_a$** — the model is *linear in strengths*. Only the
  *difference* of strengths matters, which is exactly what a goal *difference*
  should depend on.
- **$s_t \sim \mathcal{N}(0,\tau)$** — a hierarchical prior. It softly centres
  strengths at zero (fixing the "add a constant to everyone" ambiguity) and lets
  the data decide, through $\tau$, how far apart teams really are. This is
  **partial pooling**: teams with few games get pulled toward average.
- **$\sigma$** — one football match is noisy; $\sigma$ is how much the real
  margin bounces around the expected one.

That's the whole model. Now we write it in PyMC almost line-for-line with the
maths above.

In [ ]:
with pm.Model() as model:
    # --- priors ---
    tau      = pm.HalfNormal("tau", 1.0)                 # spread of team strengths
    strength = pm.Normal("strength", 0.0, tau, shape=n_teams)
    home_adv = pm.Normal("home_adv", 0.3, 0.5)           # alpha
    sigma    = pm.HalfNormal("sigma", 3.0)               # match noise

    # --- expected margin for every match (vectorised) ---
    mu = home_adv + strength[home_i] - strength[away_i]

    # --- Gaussian likelihood on the observed goal difference ---
    pm.Normal("obs", mu=mu, sigma=sigma, observed=y)

    # --- fit with NUTS (Hamiltonian Monte Carlo) ---
    idata = pm.sample(1000, tune=1000, chains=4, target_accept=0.9,
                      random_seed=RNG)

## 5. Did it converge?

Before trusting anything, check the sampler. **R-hat** should be $\approx 1.00$
(chains agree) and there should be no divergence warnings. `az.summary` reports
the posterior mean, a 94% credible interval, and R-hat.

In [ ]:
az.summary(idata, var_names=["home_adv", "sigma", "tau"]).round(3)

In [ ]:
az.plot_trace(idata, var_names=["home_adv", "sigma", "tau"])
plt.tight_layout(); plt.show()

## 6. Read the parameters

The parameters are interpretable — that is the payoff of a simple model.

- **`home_adv`** ($\alpha$): the average home edge, in goals of margin.
- **`sigma`**: the match-to-match noise on the margin.
- **`strength`**: one number per team — higher means a larger expected winning
  margin against an average opponent.

In [ ]:
strength_post = idata.posterior["strength"].mean(("chain", "draw")).to_numpy()
ranking = (pd.DataFrame({"team": teams, "strength": strength_post})
           .sort_values("strength", ascending=False).reset_index(drop=True))

top = ranking.head(12).iloc[::-1]
plt.barh(top["team"], top["strength"], color="#2b8cbe")
plt.xlabel("posterior mean strength  (expected margin vs. average team)")
plt.title("Team strength from the Gaussian goal-difference model")
plt.tight_layout(); plt.show()

ranking.head(10)

## 7. Predict a fixture's goal difference

To forecast a match we build the **posterior-predictive** distribution of the
goal difference: for every posterior draw of the parameters we compute the
expected margin $\mu$ and add Gaussian noise $\sigma$. That gives a full
distribution over the *margin* — and its **sign** gives win / draw / loss.

Because the Gaussian is continuous, we call a predicted margin within
$\pm 0.5$ a **draw**, above $+0.5$ a **home win**, below $-0.5$ an **away win**.

In [ ]:
def predict_diff(home, away, neutral=True, seed=0):
    """Posterior-predictive goal difference for home vs away."""
    post = idata.posterior
    st = post["strength"].stack(s=("chain", "draw")).to_numpy()   # (n_teams, S)
    ha = post["home_adv"].stack(s=("chain", "draw")).to_numpy()   # (S,)
    sg = post["sigma"].stack(s=("chain", "draw")).to_numpy()      # (S,)

    mu = (0.0 if neutral else ha) + st[team_idx[home]] - st[team_idx[away]]
    rng = np.random.default_rng(seed)
    diff = mu + sg * rng.standard_normal(mu.shape)                # predictive margin
    return diff


diff = predict_diff("Mexico", "South Korea", neutral=True)
summary = {
    "expected_margin": round(float(diff.mean()), 2),
    "P(home win)": round(float((diff > 0.5).mean()), 2),
    "P(draw)":     round(float((np.abs(diff) <= 0.5).mean()), 2),
    "P(away win)": round(float((diff < -0.5).mean()), 2),
}
print("Mexico vs South Korea (neutral):", summary)

plt.hist(diff, bins=40, edgecolor="white", color="#9ecae1")
plt.axvline(0, color="k", ls="--", lw=1)
plt.axvline(diff.mean(), color="#d94801", lw=2, label=f"E[margin]={diff.mean():.2f}")
plt.xlabel("predicted goal difference  (home − away)")
plt.title("Posterior-predictive margin: Mexico vs South Korea")
plt.legend(); plt.tight_layout(); plt.show()

## 8. Takeaways — and how this differs from the scoreline model

**What we built.** A one-line Bayesian idea — *margin = home edge + strength gap
+ noise* — that (a) targets the **signed goal difference** so the sign is the
result, (b) uses a **Gaussian**, making it a clean, interpretable linear model,
and (c) still gives full win/draw/loss probabilities from the predictive
distribution.

**Gaussian goal-difference vs. Poisson scorelines (the project's main model):**

| | This notebook (Gaussian on the margin) | Main model (Poisson on goals) |
|---|---|---|
| Target | goal **difference** (one signed number) | each team's **goals** (two counts) |
| Gives | margin, win/draw/loss | full **scoreline** grid → 1X2, over/under, both-teams-score |
| Pros | dead simple, interpretable, models the result directly | respects that goals are counts; richer outputs |
| Cons | ignores that goals are integer counts; no scoreline | more moving parts |

**When to prefer this one:** teaching, a quick baseline, or when you only need
the margin/result. **When to prefer Poisson:** when you need scorelines or exact
totals, or to simulate a whole tournament of specific results.

**Natural next steps to try in this notebook**
- Add back the home advantage for host nations only (`neutral=False`).
- Replace `Normal` strengths with `pm.ZeroSumNormal` for exact identifiability.
- Compare predicted vs. actual sign on a held-out set (a tiny accuracy check).